In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import torch.nn.functional as F
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt

import time

In [ ]:
# --- Check for GPU ---
# This will be important when you move to your cluster
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
#################################################################
# 1. AGENT BRAIN CLASS (As designed)
#################################################################

class AgentBrain(nn.Module):
    """
    An Actor-Critic agent with a shared RNN trunk (GRU).
    """
    def __init__(self, public_input_size, private_input_size, rnn_hidden_size, 
                 move_action_size, pull_action_size, gaze_action_size):
        
        super(AgentBrain, self).__init__()
        
        self.input_size = public_input_size + private_input_size
        self.rnn_hidden_size = rnn_hidden_size
        self.move_action_size = move_action_size
        self.pull_action_size = pull_action_size
        self.gaze_action_size = gaze_action_size

        # --- 1. The Shared RNN "Trunk" ---
        self.rnn_trunk = nn.GRUCell(self.input_size, self.rnn_hidden_size)

        # --- 2. The "Actor" Head ---
        self.move_head = nn.Linear(self.rnn_hidden_size, move_action_size)
        self.pull_head = nn.Linear(self.rnn_hidden_size, pull_action_size)
        self.gaze_head = nn.Linear(self.rnn_hidden_size, gaze_action_size)

        # --- 3. The "Critic" Head ---
        self.critic_head = nn.Linear(self.rnn_hidden_size, 1)

    def forward(self, public_input, private_gaze_input, prev_hidden_state, pull_mask):
        
        # --- Build the Full Input State ---
        full_input = torch.cat([public_input, private_gaze_input], dim=-1)

        # --- Run the "Accumulator" (RNN Trunk) ---
        current_hidden_state = self.rnn_trunk(full_input, prev_hidden_state)

        # --- Get Actor & Critic Outputs ---
        value = self.critic_head(current_hidden_state)
        
        move_logits = self.move_head(current_hidden_state)
        gaze_logits = self.gaze_head(current_hidden_state)
        
        # --- NEW: Apply Action Mask ---
        pull_logits = self.pull_head(current_hidden_state)
        # By adding -inf (or a large negative number), the softmax probability
        # for the masked action (action 1, "pull") becomes zero.
        pull_logits = pull_logits + pull_mask
        # --- END NEW ---
        
        # Return logits, value, and the new hidden state
        return (move_logits, pull_logits, gaze_logits), value, current_hidden_state

### different environment settings

In [ ]:
#################################################################
# 2. MASTER CURRICULUM ENVIRONMENT (v16 - Hard Rule + Stay Action)
#################################################################
class CurriculumEnv:
    """
    Continuous, non-episodic environment (v16 - Hard Rule + Stay Action)
    
    - *** NEW V16 ***
    - This is the "best of both worlds" environment.
    - 1. It includes a 5th move action (index 4) for "Stay Still",
    -    which the agent can learn to use for general waiting.
    - 2. It ALSO enforces a "hard rule": if action['pull'] == 1,
    -    the agent's move action is IGNORED and it stays still.
    -
    - This means the agent does NOT have to learn to coordinate
    - (move==4) and (pull==1). It just has to learn (pull==1).
    -
    - (Other curriculum logic remains the same)
    """
    
    def __init__(self, grid_size_x=12, grid_size_y=12):
        self.grid_size_x = grid_size_x
        self.grid_size_y = grid_size_y
        self.boundary_x = self.grid_size_x // 2 
        self.locations = {
            'lever_A': np.array([2, 10]), 'port_A':  np.array([2, 1]),
            'lever_B': np.array([9, 10]), 'port_B':  np.array([9, 1]),
        }
        self.pull_reset_time = 20 # was 20
        self.training_phase = "SELF_REWARD" 
        self.current_sync_window = 30 
        self.reset() 

    def set_training_phase(self, phase, sync_window=30):
        self.training_phase = phase
        self.current_sync_window = sync_window
        print(f"*** ENV UPDATED: Phase={self.training_phase}, SyncWindow={self.current_sync_window} ***")

    def _get_state(self):
        
        lever_A_state = 0 # 0=Ready
        lever_B_state = 0 # 0=Ready
            
        state_A = {
            'self_pos': self.agent_A_pos,
            'lever_pos': self.locations['lever_A'],
            'port_pos': self.locations['port_A'],
            'self_lever_state': lever_A_state,
            'self_port_state': self.port_A_state,
            'can_pull': self.agent_A_pull_cooldown == 0,
            'self_is_drinking': self.A_is_drinking, # <-- ADD THIS LINE
            'success_sound': self.success_sound_played 
        }
        state_B = {
            'self_pos': self.agent_B_pos,
            'lever_pos': self.locations['lever_B'],
            'port_pos': self.locations['port_B'],
            'self_lever_state': lever_B_state,
            'self_port_state': self.port_B_state,
            'can_pull': self.agent_B_pull_cooldown == 0,
            'self_is_drinking': self.B_is_drinking, # <-- ADD THIS LINE
            'success_sound': self.success_sound_played
        }
        return state_A, state_B

    def reset(self):
        self.agent_A_pos = self.locations['port_A'].copy()
        self.agent_B_pos = self.locations['port_B'].copy()
        self.current_step = 0 
        self.lever_A_state = 0
        self.lever_B_state = 0
        self.pull_A_timestamp = -1 
        self.pull_B_timestamp = -1 
        self.port_A_state = 0
        self.port_B_state = 0
        self.agent_A_pull_cooldown = 0
        self.agent_B_pull_cooldown = 0
        self.success_sound_played = 0
        self.A_time_since_pull = 1000
        self.B_time_since_pull = 1000
        
        # --- NEW STATES FOR DRINKING ---
        self.A_is_drinking = False
        self.B_is_drinking = False
        self.A_drink_timer = 0
        self.B_drink_timer = 0
        # --- END NEW STATES ---
        
        return self._get_state()

    def _move(self, current_pos, move_action, agent_id):
        # (This is from v15 - it understands action 4)
        if move_action == -1: return current_pos
        if move_action == 0: current_pos[1] += 1
        elif move_action == 1: current_pos[1] -= 1
        elif move_action == 2: current_pos[0] -= 1
        elif move_action == 3: current_pos[0] += 1
        elif move_action == 4: pass # Action 4 is "Stay Still"
            
        current_pos[1] = np.clip(current_pos[1], 0, self.grid_size_y - 1)
        if agent_id == 'A':
            current_pos[0] = np.clip(current_pos[0], 0, self.boundary_x - 1)
        elif agent_id == 'B':
            current_pos[0] = np.clip(current_pos[0], self.boundary_x, self.grid_size_x - 1)
        return current_pos

    def step(self, action_A, action_B):
        
        
        # --- NEW DRINKING LOGIC ---
        if self.A_drink_timer > 0:
            self.A_drink_timer -= 1
            if self.A_drink_timer == 0: self.A_is_drinking = False
            action_A['move'] = 4 # Force "Stay Still"
        
        if self.B_drink_timer > 0:
            self.B_drink_timer -= 1
            if self.B_drink_timer == 0: self.B_is_drinking = False
            action_B['move'] = 4 # Force "Stay Still"
        # --- END NEW LOGIC ---
        
        self.current_step += 1
        reward_A, reward_B = -0.01, -0.01 
        self.success_sound_played = 0 
        done = False
        coordination_attempt_complete = False 

        self.A_time_since_pull += 1
        self.B_time_since_pull += 1

        if self.agent_A_pull_cooldown > 0: self.agent_A_pull_cooldown -= 1
        if self.agent_B_pull_cooldown > 0: self.agent_B_pull_cooldown -= 1
        
        
        # -----------------------------------------------------------------
        # --- *** NEW: "HARD RULE" MOVEMENT LOGIC (V16) *** ---
        # -----------------------------------------------------------------
        
        # --- Agent A Movement ---
        target_A = self.locations['port_A'] if self.port_A_state == 1 else self.locations['lever_A']
        old_dist_A = np.linalg.norm(self.agent_A_pos - target_A)
        
        if action_A['pull'] == 1:
            # HARD RULE: If pulling, DO NOT move, regardless of move action.
            pass 
        else:
            # If not pulling, process the agent's chosen move action (0-4).
            self.agent_A_pos = self._move(self.agent_A_pos, action_A['move'], 'A')
        
        new_dist_A = np.linalg.norm(self.agent_A_pos - target_A)
        if new_dist_A < old_dist_A:
            reward_A += 0.02
        
        # --- Agent B Movement ---
        target_B = self.locations['port_B'] if self.port_B_state == 1 else self.locations['lever_B']
        old_dist_B = np.linalg.norm(self.agent_B_pos - target_B)
        
        if action_B['pull'] == 1:
            # HARD RULE: If pulling, DO NOT move, regardless of move action.
            pass
        else:
            # If not pulling, process the agent's chosen move action (0-4).
            self.agent_B_pos = self._move(self.agent_B_pos, action_B['move'], 'B')

        new_dist_B = np.linalg.norm(self.agent_B_pos - target_B)
        if new_dist_B < old_dist_B:
            reward_B += 0.02
        # -----------------------------------------------------------------
        # --- *** END NEW *** ---
        # -----------------------------------------------------------------


        # Check location *after* movement (or non-movement) has been processed
        at_lever_A = np.array_equal(self.agent_A_pos, self.locations['lever_A'])
        at_lever_B = np.array_equal(self.agent_B_pos, self.locations['lever_B'])

        if self.training_phase == "SELF_REWARD":
            if at_lever_A:
                reward_A += 1.0
            if at_lever_B:
                reward_B += 1.0

            # --- Agent A Pull Logic ---
            # This check is now guaranteed to pass if the agent was at the
            # lever, because the "hard rule" prevented it from moving.
            if (action_A['pull'] == 1 and at_lever_A):
                self.A_time_since_pull = 0
                reward_A += 10.0 
                self.port_A_state = 1 
                self.success_sound_played = 1
                self.agent_A_pull_cooldown = self.pull_reset_time

            # --- Agent B Pull Logic ---
            if (action_B['pull'] == 1 and at_lever_B):
                self.B_time_since_pull = 0
                reward_B += 10.0
                self.port_B_state = 1
                self.success_sound_played = 1
                self.agent_B_pull_cooldown = self.pull_reset_time

        else: # "COOP_ASSISTED" or "COOP_FULL"
            
            # --- Agent A Gaze Reward Logic ---
            if 0:
                # A gets reward if it looks at B AND B is "actionable" (Ready or Waiting)
                B_is_ready = (at_lever_B and self.agent_B_pull_cooldown == 0)
                B_is_waiting = (self.pull_B_timestamp != -1) # B pulled and is waiting for A

                if action_A['gaze'] == 1 and (B_is_ready or B_is_waiting):
                    reward_A += 1.0

                # --- Agent B Gaze Reward Logic ---
                # B gets reward if it looks at A AND A is "actionable" (Ready or Waiting)
                A_is_ready = (at_lever_A and self.agent_A_pull_cooldown == 0)
                A_is_waiting = (self.pull_A_timestamp != -1) # A pulled and is waiting for B

                if action_B['gaze'] == 1 and (A_is_ready or A_is_waiting):
                    reward_B += 1.0
            
            if 0:
                if at_lever_A and at_lever_B:
                    reward_A += 1.0 
                    reward_B += 1.0 

            # --- Agent A pull attempt ---
            if (action_A['pull'] == 1 and at_lever_A):
                self.A_time_since_pull = 0
                if self.agent_A_pull_cooldown > 0:
                    reward_A -= 0.5
                    self.agent_A_pull_cooldown = self.pull_reset_time 
                else:
                    self.pull_A_timestamp = self.current_step
                    self.agent_A_pull_cooldown = self.pull_reset_time
                    if self.pull_B_timestamp != -1: 
                        coordination_attempt_complete = True
            
            # --- Agent B pull attempt ---
            if (action_B['pull'] == 1 and at_lever_B):
                self.B_time_since_pull = 0
                if self.agent_B_pull_cooldown > 0:
                    reward_B -= 0.5
                    self.agent_B_pull_cooldown = self.pull_reset_time 
                else:
                    self.pull_B_timestamp = self.current_step
                    self.agent_B_pull_cooldown = self.pull_reset_time
                    if self.pull_A_timestamp != -1: 
                        coordination_attempt_complete = True
            
            if coordination_attempt_complete:
                if abs(self.pull_A_timestamp - self.pull_B_timestamp) <= self.current_sync_window:
                    # SUCCESS
                    if self.training_phase == "COOP_ASSISTED":
                        reward_A += 10.0; reward_B += 10.0
                    else: # COOP_FULL
                        reward_A += 10.0; reward_B += 10.0
                    self.port_A_state = 1; self.port_B_state = 1 
                    self.success_sound_played = 1 
                
                else:
                    # FAILED COORD
                    if self.training_phase == "COOP_ASSISTED":
                        reward_A += 10.0; reward_B += 10.0
                        self.port_A_state = 1; self.port_B_state = 1
                    else: # COOP_FULL
                        reward_A -= 0.5
                        reward_B -= 0.5
                
                self.pull_A_timestamp = -1
                self.pull_B_timestamp = -1

        # --- Reward Consumption (Instantaneous) ---
        if (np.array_equal(self.agent_A_pos, self.locations['port_A']) and
            self.port_A_state == 1 and not self.A_is_drinking): # <-- ADDED CHECK
            
            reward_A += 15.0 
            self.port_A_state = 0 
            
            # --- START DRINKING ---
            self.A_is_drinking = True
            self.A_drink_timer = np.random.randint(5, 30) # Random 5-30 steps
            
        if (np.array_equal(self.agent_B_pos, self.locations['port_B']) and
            self.port_B_state == 1 and not self.B_is_drinking): # <-- ADDED CHECK
            
            reward_B += 15.0
            self.port_B_state = 0
            
            # --- START DRINKING ---
            self.B_is_drinking = True
            self.B_drink_timer = np.random.randint(5, 30) # Random 5-30 steps
        
        next_state_A, next_state_B = self._get_state()
        
        return next_state_A, next_state_B, reward_A, reward_B, done

### helper functions

In [ ]:
#################################################################
# 3. HELPER FUNCTIONS (Winning Setup - Run 4)
#################################################################

def preprocess_state(state_dict, grid_size_x, grid_size_y):
    """
    Converts the public state dictionary into a flat tensor.
    --- USES 'can_pull' BOOLEAN AND EXCLUDES PARTNER'S POSITION ---
    """
    self_pos = state_dict['self_pos'] / np.array([grid_size_x, grid_size_y])
    # partner_pos = ... # <-- REMOVED
    lever_pos = state_dict['lever_pos'] / np.array([grid_size_x, grid_size_y])
    port_pos = state_dict['port_pos'] / np.array([grid_size_x, grid_size_y])
    
    lever_state = F.one_hot(torch.tensor(state_dict['self_lever_state']), num_classes=3).float()
    port_state = F.one_hot(torch.tensor(state_dict['self_port_state']), num_classes=2).float()
    
    # --- This is the correct boolean signal ---
    can_pull = torch.tensor([1.0 if state_dict['can_pull'] else 0.0])
    
    # --- THIS IS THE NEW LINE YOU MUST ADD ---
    is_drinking = torch.tensor([1.0 if state_dict['self_is_drinking'] else 0.0]) # <-- NEW
    
    success_sound = torch.tensor([1.0 if state_dict['success_sound'] else 0.0])
    
    tensor_state = torch.cat([\
        torch.from_numpy(self_pos).float(),
        # torch.from_numpy(partner_pos).float(), # <-- REMOVED
        torch.from_numpy(lever_pos).float(),
        torch.from_numpy(port_pos).float(),
        lever_state,
        port_state,
        is_drinking, # <-- ADD THE NEW TENSOR HERE
        can_pull,
        success_sound
    ], dim=0)
    
    return tensor_state.unsqueeze(0).to(device) # Add batch dim

def preprocess_partner_pos(partner_pos_array, grid_size_x, grid_size_y):
    """ NEW: Helper to normalize the partner's position for the private input """
    pos_tensor = torch.from_numpy(partner_pos_array).float()
    pos_normalized = pos_tensor / torch.tensor([grid_size_x, grid_size_y]).float()
    return pos_normalized.unsqueeze(0).to(device) # Add batch dim


def sample_actions(logits_tuple):
    # (This function is the same)
    move_logits, pull_logits, gaze_logits = logits_tuple
    move_dist = Categorical(logits=move_logits)
    move_action = move_dist.sample()
    move_log_prob = move_dist.log_prob(move_action)
    pull_dist = Categorical(logits=pull_logits)
    pull_action = pull_dist.sample()
    pull_log_prob = pull_dist.log_prob(pull_action)
    gaze_dist = Categorical(logits=gaze_logits)
    gaze_action = gaze_dist.sample()
    gaze_log_prob = gaze_dist.log_prob(gaze_action)
    total_log_prob = move_log_prob + pull_log_prob + gaze_log_prob
    actions_dict = {
        'move': move_action.item(),
        'pull': pull_action.item(),
        'gaze': gaze_action.item()
    }
    return actions_dict, total_log_prob

### main training function

In [ ]:
#################################################################
# 4. TRAINING HYPERPARAMETERS (NEW)
#################################################################

# --- Environment ---
# GRID_X = 8 # 20
# GRID_Y = 10 # 20
GRID_X = 12
GRID_Y = 12
SESSION_LENGTH = 200 # Short session for prototyping
EPISODE_LENGTH = 200     # Max steps per trial

# --- Model ---
RNN_HIDDEN_SIZE = 128 # 64

# --- NEW INPUT SIZES ---
# 2(pos) + 2(lever) + 2(port) + 3(lever_state) + 2(port_state)  + 1(pull) + 1(sound) = 14
PUBLIC_INPUT_SIZE = 14 
# Partner's position (2) + Partner's can_pull (1) + Partner's time_since_pull (1) = 4
PRIVATE_INPUT_SIZE = 4

MOVE_ACTIONS = 5  # U, D, L, R, Stay Still
PULL_ACTIONS = 2  # No, Yes
GAZE_ACTIONS = 2  # No, Yes

# --- Training ---
LR = 1e-4                # Learning rate
GAMMA = 0.99             # Discount factor for future rewards
N_STEPS = 500            # (WAS 500) Shorter rollout for faster updates
TOTAL_TRAIN_STEPS = 2000000  # <-- INCREASED FROM 1.4M
ENTROPY_COEF = 0.0001     # (WAS 0.0001) Encourage exploration!
VALUE_LOSS_COEF = 1.0    # (WAS 0.5) Force the Critic to learn faster

In [ ]:
#################################################################
# 5. THE MASTER TRAINING LOOP (UPDATED)
#################################################################
def train():
    print("Starting training with FULL 6-PHASE CURRICULUM (Gaze-Ready + Action Masking)...\n") # <-- Updated print
    
    env = CurriculumEnv(GRID_X, GRID_Y)
    
    agent_A = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)
    agent_B = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)

    optimizer_A = optim.Adam(agent_A.parameters(), lr=LR)
    optimizer_B = optim.Adam(agent_B.parameters(), lr=LR)

    # --- (Curriculum Timings are the same) ---
    CURRICULUM_TIMINGS = {
        # --- Phase A: "Always On" (Learn the VALUE of info) ---
        0:        ("SELF_REWARD", 200, "ALWAYS_ON"),
        200000:   ("COOP_ASSISTED", 200, "ALWAYS_ON"),
        400000:   ("COOP_FULL", 200, "ALWAYS_ON"),
        500000:   ("COOP_FULL", 100, "ALWAYS_ON"),
        600000:   ("COOP_FULL", 50, "ALWAYS_ON"),
        800000:   ("COOP_FULL", 30, "ALWAYS_ON"),  
        1000000:  ("COOP_FULL", 20, "ALWAYS_ON"),
        
        # --- Phase B: "Filtered" (Learn the ACTION of gazing) ---
        1300000:  ("COOP_FULL", 20, "FILTERED"),
        1600000:  ("COOP_FULL", 10, "FILTERED")
    }
    
    # --- Get Initial States (Called ONCE) ---
    env.set_training_phase("SELF_REWARD", sync_window=200)
    state_A_dict, state_B_dict = env.reset()
    
    public_A = preprocess_state(state_A_dict, GRID_X, GRID_Y)
    public_B = preprocess_state(state_B_dict, GRID_X, GRID_Y)
    hidden_A = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)
    hidden_B = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)
    private_A = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device) # <-- Size 4
    private_B = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device) # <-- Size 4

    # --- (Log init is the same) ---
    full_training_log = []
    
    for update_step in range(TOTAL_TRAIN_STEPS // N_STEPS):
        
        total_steps_so_far = update_step * N_STEPS
        
        ## --- (Curriculum update is the same) ---
        current_gaze_mode = "FILTERED" # Default
        for milestone_step in sorted(CURRICULUM_TIMINGS.keys(), reverse=True):
            if total_steps_so_far >= milestone_step:
                # --- NEW: Now unpacks 3 curriculum values ---
                phase, window, gaze_mode = CURRICULUM_TIMINGS[milestone_step]
                current_gaze_mode = gaze_mode # Store the current mode
                # --- END NEW ---
                if env.training_phase != phase or env.current_sync_window != window:
                    env.set_training_phase(phase, sync_window=window)
                break
        
        # --- (Rollout init is the same) ---
        all_log_probs_A, all_values_A, all_rewards_A = [], [], []
        all_log_probs_B, all_values_B, all_rewards_B = [], [], []
        all_entropy_A, all_entropy_B = 0, 0

        hidden_A = hidden_A.detach()
        hidden_B = hidden_B.detach()

        for step_in_rollout in range(N_STEPS):
            
            # -----------------------------------------------------------
            # --- *** NEW: CREATE ACTION MASKS *** ---
            # -----------------------------------------------------------
            # Check conditions based on the *current* state
            at_lever_A = np.array_equal(state_A_dict['self_pos'], env.locations['lever_A'])
            at_lever_B = np.array_equal(state_B_dict['self_pos'], env.locations['lever_B'])

            can_pull_A = state_A_dict['can_pull'] # This is env.agent_A_pull_cooldown == 0
            can_pull_B = state_B_dict['can_pull'] # This is env.agent_B_pull_cooldown == 0

            # Mask is [0., -1e9] if action 1 ("pull") is NOT allowed
            # Mask is [0., 0.] if action 1 ("pull") IS allowed
            
            if can_pull_A and at_lever_A:
                mask_A = torch.tensor([[0., 0.]], device=device).float()
            else:
                mask_A = torch.tensor([[0., -1e9]], device=device).float()
                
            if can_pull_B and at_lever_B:
                mask_B = torch.tensor([[0., 0.]], device=device).float()
            else:
                mask_B = torch.tensor([[0., -1e9]], device=device).float()
            # -----------------------------------------------------------
            # --- *** END NEW *** ---
            # -----------------------------------------------------------

            
            # --- (Agent runs now pass the mask) ---
            (logits_A_tuple), value_A, next_hidden_A = agent_A(public_A, private_A, hidden_A, mask_A)
            actions_A_dict, log_prob_A = sample_actions(logits_A_tuple)
            entropy_A = -sum((F.softmax(l, dim=-1) * F.log_softmax(l, dim=-1)).sum() for l in logits_A_tuple)
            all_entropy_A += entropy_A
            
            (logits_B_tuple), value_B, next_hidden_B = agent_B(public_B, private_B, hidden_B, mask_B)
            actions_B_dict, log_prob_B = sample_actions(logits_B_tuple)
            entropy_B = -sum((F.softmax(l, dim=-1) * F.log_softmax(l, dim=-1)).sum() for l in logits_B_tuple)
            all_entropy_B += entropy_B

            # --- (Env step is the same) ---
            next_state_A_dict, next_state_B_dict, reward_A, reward_B, done = \
                env.step(actions_A_dict, actions_B_dict)
                
            # --- (Logging is the same) ---
            log_step = total_steps_so_far + step_in_rollout
            full_training_log.append({
                'step': log_step,
                'phase': env.training_phase,
                'window': env.current_sync_window,
                'A_pos_x': env.agent_A_pos[0],
                'A_pos_y': env.agent_A_pos[1],
                'A_action_move': actions_A_dict['move'],
                'A_action_pull': actions_A_dict['pull'],
                'A_action_gaze': actions_A_dict['gaze'],
                'A_reward': reward_A, 
                'A_value': value_A.item(),  # <-- ADD THIS LINE
                'B_pos_x': env.agent_B_pos[0],
                'B_pos_y': env.agent_B_pos[1],
                'B_action_move': actions_B_dict['move'],
                'B_action_pull': actions_B_dict['pull'],
                'B_action_gaze': actions_B_dict['gaze'],
                'B_reward': reward_B, 
                'B_value': value_B.item(),  # <-- AND THIS LINE
            })

            # --- (Storing rewards/logs is the same) ---
            all_log_probs_A.append(log_prob_A)
            all_values_A.append(value_A)
            all_rewards_A.append(reward_A)
            all_log_probs_B.append(log_prob_B)
            all_values_B.append(value_B)
            all_rewards_B.append(reward_B)

            # --- (NEW Gaze Tensor logic is now conditional) ---
            partner_pos_B_tensor = preprocess_partner_pos(env.agent_B_pos, GRID_X, GRID_Y)
            partner_pos_A_tensor = preprocess_partner_pos(env.agent_A_pos, GRID_X, GRID_Y)
            
            partner_A_can_pull = torch.tensor([[
                1.0 if next_state_A_dict['can_pull'] else 0.0
            ]], device=device).float()
            partner_B_can_pull = torch.tensor([[
                1.0 if next_state_B_dict['can_pull'] else 0.0
            ]], device=device).float()

            partner_A_time_norm = torch.tensor([[
                min(env.A_time_since_pull / env.current_sync_window, 1.0)
            ]], device=device).float()
            partner_B_time_norm = torch.tensor([[
                min(env.B_time_since_pull / env.current_sync_window, 1.0)
            ]], device=device).float()

            if current_gaze_mode == "ALWAYS_ON":
                # Agents get full info for free, regardless of gaze action
                private_A = torch.cat([partner_pos_B_tensor, partner_B_can_pull, partner_B_time_norm], dim=-1)
                private_B = torch.cat([partner_pos_A_tensor, partner_A_can_pull, partner_A_time_norm], dim=-1)
            
            else: # "FILTERED" mode (the original logic)
                if actions_A_dict['gaze'] == 1:
                    private_A = torch.cat([partner_pos_B_tensor, partner_B_can_pull, partner_B_time_norm], dim=-1)
                else:
                    private_A = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device) 
                
                if actions_B_dict['gaze'] == 1:
                    private_B = torch.cat([partner_pos_A_tensor, partner_A_can_pull, partner_A_time_norm], dim=-1)
                else:
                    private_B = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
            # --- (End Gaze Tensor logic) ---


            public_A = preprocess_state(next_state_A_dict, GRID_X, GRID_Y)
            public_B = preprocess_state(next_state_B_dict, GRID_X, GRID_Y)
            hidden_A = next_hidden_A
            hidden_B = next_hidden_B

            # --- *** NEW: Update state_dicts for next loop's mask *** ---
            state_A_dict = next_state_A_dict
            state_B_dict = next_state_B_dict
            # --- *** END NEW *** ---

        # --- (Learning/Backprop is the same) ---
        _, next_value_A, _ = agent_A(public_A, private_A, hidden_A, mask_A) # Pass mask for final value
        _, next_value_B, _ = agent_B(public_B, private_B, hidden_B, mask_B) # Pass mask for final value

        rewards_A = torch.tensor(all_rewards_A).float().to(device)
        values_A = torch.cat(all_values_A)
        log_probs_A = torch.cat(all_log_probs_A)
        returns_A = torch.zeros_like(rewards_A)
        R_A = next_value_A.detach()
        for t in reversed(range(len(rewards_A))):
            R_A = rewards_A[t] + GAMMA * R_A
            returns_A[t] = R_A
        advantages_A = returns_A - values_A.detach() 
        actor_loss_A = -(log_probs_A * advantages_A).mean()
        critic_loss_A = F.mse_loss(values_A.squeeze(), returns_A)
        loss_A = (actor_loss_A + VALUE_LOSS_COEF * critic_loss_A - ENTROPY_COEF * all_entropy_A)

        rewards_B = torch.tensor(all_rewards_B).float().to(device)
        values_B = torch.cat(all_values_B)
        log_probs_B = torch.cat(all_log_probs_B)
        returns_B = torch.zeros_like(rewards_B)
        R_B = next_value_B.detach()
        for t in reversed(range(len(rewards_B))):
            R_B = rewards_B[t] + GAMMA * R_B
            returns_B[t] = R_B
        advantages_B = returns_B - values_B.detach()
        actor_loss_B = -(log_probs_B * advantages_B).mean()
        critic_loss_B = F.mse_loss(values_B.squeeze(), returns_B)
        loss_B = (actor_loss_B + VALUE_LOSS_COEF * critic_loss_B - ENTROPY_COEF * all_entropy_B)

        # --- (Optimization step is the same) ---
        optimizer_A.zero_grad()
        loss_A.backward()
        torch.nn.utils.clip_grad_norm_(agent_A.parameters(), 1.0)
        optimizer_A.step()
        optimizer_B.zero_grad()
        loss_B.backward()
        torch.nn.utils.clip_grad_norm_(agent_B.parameters(), 1.0)
        optimizer_B.step()
        
        # --- (Printing is the same) ---
        if update_step % (5000 // N_STEPS) == 0: 
            total_steps_so_far = update_step * N_STEPS
            total_rewards_A = sum(all_rewards_A)
            total_rewards_B = sum(all_rewards_B)
            print(f"Step: {total_steps_so_far}, Phase: {env.training_phase}, Win: {env.current_sync_window}, "
                  f"R_A: {total_rewards_A:.2f}, R_B: {total_rewards_B:.2f}, "
                  f"Loss_A: {loss_A.item():.4f}, Loss_B: {loss_B.item():.4f}")

    print("Training complete.")
    
    # --- (Saving log is the same) ---
    print("Converting log to DataFrame...")
    df = pd.DataFrame(full_training_log)
    df.to_pickle("full_training_log_new.pkl")
    print("Full training log saved to 'full_training_log_new.pkl'")
    
    torch.save(agent_A.state_dict(), "agent_A_brain_new.pth")
    torch.save(agent_B.state_dict(), "agent_B_brain_new.pth")
    

# --- To run the training ---
if 1:
    if __name__ == "__main__":
        train()
else:
    print('no training, will load existing trained data')

In [ ]:
# load and analysis the data from the training process
df_training_log = pd.read_pickle("full_training_log_new.pkl")
df = df_training_log


In [ ]:
df

In [ ]:
np.unique(df['B_reward'])

In [ ]:
plt.plot(df['A_pos_y'])

In [ ]:
print("Calculating pull efficiency...")
import seaborn as sns


# Use the 100-step bin as requested
bin_size = 100
df['step_bin'] = (df['step'] // bin_size) * bin_size

# Define "attempted" vs "successful" pulls
# A "successful" pull is one that resulted in a large reward (> 5.0)
df['A_attempted_pull'] = (df['A_action_pull'] == 1)
df['B_attempted_pull'] = (df['B_action_pull'] == 1)
df['A_successful_pull'] = (df['A_action_pull'] == 1) & (df['A_reward'] > 5.0)
df['B_successful_pull'] = (df['B_action_pull'] == 1) & (df['B_reward'] > 5.0)


# --- 3. Aggregate Data ---
# Group by the 100-step bins and sum up the attempts and successes
df_agg = df.groupby('step_bin').agg(
    A_attempted_pulls=('A_attempted_pull', 'sum'),
    A_successful_pulls=('A_successful_pull', 'sum'),
    B_attempted_pulls=('B_attempted_pull', 'sum'),
    B_successful_pulls=('B_successful_pull', 'sum'),
    phase=('phase', 'first'),
    window=('window', 'first')
).reset_index()

# --- 4. Calculate Success Rate (Pull Efficiency) ---
# Calculate the raw success rate for each bin
# We use .fillna(0) to handle bins with 0 attempts (0/0 = NaN)
df_agg['A_success_rate'] = (df_agg['A_successful_pulls'] / df_agg['A_attempted_pulls']).fillna(0)
df_agg['B_success_rate'] = (df_agg['B_successful_pulls'] / df_agg['B_attempted_pulls']).fillna(0)

# --- 5. Smooth the Data (Rolling Average) ---
# The raw 100-step rate is too noisy. Let's smooth it with a 10,000-step
# (100 bins) rolling average to see the trend.
rolling_window_size = 100 
df_agg['A_rate_rolling'] = df_agg['A_success_rate'].rolling(window=rolling_window_size, min_periods=1).mean()
df_agg['B_rate_rolling'] = df_agg['B_success_rate'].rolling(window=rolling_window_size, min_periods=1).mean()

print("Calculation complete. Plotting...")

# --- 6. Plotting ---
plt.figure(figsize=(20, 10))
sns.lineplot(data=df_agg, x='step_bin', y='A_rate_rolling', label='Agent A: Pull Efficiency (10k-step avg)')
sns.lineplot(data=df_agg, x='step_bin', y='B_rate_rolling', label='Agent B: Pull Efficiency (10k-step avg)')

plt.title('Pull Efficiency Over Training (Successful Pulls / Total Attempted Pulls)', fontsize=18)
plt.ylabel('Pull Efficiency (Rolling Average)', fontsize=14)
plt.xlabel('Training Step', fontsize=14)
plt.legend(loc='upper left', fontsize=12)

# --- Add vertical lines for curriculum changes ---
phase_changes = df_agg.drop_duplicates('phase')[['step_bin', 'phase']].iloc[1:]
window_changes = df_agg.drop_duplicates('window')[['step_bin', 'window']].iloc[1:]

ax = plt.gca()
for _, row in phase_changes.iterrows():
    ax.axvline(x=row['step_bin'], color='red', linestyle='--', linewidth=2)
    ax.text(row['step_bin'] + 5000, 0.095, f"Phase: {row['phase']}", 
            color='red', rotation=90, ha='left', va='top', fontsize=12)

for _, row in window_changes.iterrows():
    ax.axvline(x=row['step_bin'], color='blue', linestyle=':', linewidth=2)
    ax.text(row['step_bin'] + 5000, 0.07, f"Window: {row['window']}", 
            color='blue', rotation=90, ha='left', va='top', fontsize=12)

plt.ylim(0, 1) # Efficiency is a rate from 0 to 1
plt.savefig("pull_efficiency_plot.pdf")

In [ ]:
# Set plotting style
sns.set(style="darkgrid")
plt.rcParams['figure.figsize'] = (18, 8)


# --- 2. Get Curriculum Change Points ---
# We do this first, on the full dataframe
bin_size = 5000
df['step_bin'] = (df['step'] // bin_size) * bin_size
df_agg_bins = df.groupby('step_bin').first().reset_index()
phase_changes = df_agg_bins.drop_duplicates('phase')[['step', 'phase']].iloc[1:]
window_changes = df_agg_bins.drop_duplicates('window')[['step', 'window']].iloc[1:]


# --- 3. Feature Engineering: Find Successful Pulls ---
print("Calculating Inter-Pull Intervals (IPI)...")

# A "successful" pull is one that resulted in a large reward (> 5.0)
df['A_successful_pull'] = (df['A_action_pull'] == 1)  & (df['A_reward'] > 5.0)
df['B_successful_pull'] = (df['B_action_pull'] == 1)  & (df['B_reward'] > 5.0)

# --- 4. Calculate IPI ---
# Create new, "sparse" DataFrames containing only the successful pulls
df_A_pulls = df[df['A_successful_pull']].copy()
df_B_pulls = df[df['B_successful_pull']].copy()

# Calculate the difference (in steps) between each successful pull
df_A_pulls['A_IPI'] = df_A_pulls['step'].diff()
df_B_pulls['B_IPI'] = df_B_pulls['step'].diff()

# --- 5. Smooth the Data (Rolling Average) ---
# Calculate the rolling average of the last 50 IPIs to see the trend
rolling_window_pulls = 50
df_A_pulls['A_IPI_rolling'] = df_A_pulls['A_IPI'].rolling(window=rolling_window_pulls, min_periods=1).mean()
df_B_pulls['B_IPI_rolling'] = df_B_pulls['B_IPI'].rolling(window=rolling_window_pulls, min_periods=1).mean()

print("Calculation complete. Plotting...")

# --- 6. Plotting ---
plt.figure(figsize=(20, 10))
sns.lineplot(data=df_A_pulls, x='step', y='A_IPI_rolling', label=f'Agent A: Avg. IPI (last {rolling_window_pulls} pulls)')
sns.lineplot(data=df_B_pulls, x='step', y='B_IPI_rolling', label=f'Agent B: Avg. IPI (last {rolling_window_pulls} pulls)')

# Add a line for the "optimal" pull interval (60-step cooldown + 5-step personal)
plt.axhline(y=65, color='gray', linestyle=':', linewidth=2, label='Optimal IPI (65 steps)')

plt.title('Inter-Pull Interval (IPI) Over Training', fontsize=18)
plt.ylabel('Average Interval (steps)', fontsize=14)
plt.xlabel('Training Step', fontsize=14)
plt.legend(loc='upper left', fontsize=12)
plt.ylim(0,400) # Zoom in on a reasonable IPI range

# --- Add vertical lines for curriculum changes ---
ax = plt.gca()
for _, row in phase_changes.iterrows():
    ax.axvline(x=row['step'], color='red', linestyle='--', linewidth=2)
    ax.text(row['step'] + 5000, 390, f"Phase: {row['phase']}", 
            color='red', rotation=90, ha='left', va='top', fontsize=12)

for _, row in window_changes.iterrows():
    ax.axvline(x=row['step'], color='blue', linestyle=':', linewidth=2)
    ax.text(row['step'] + 5000, 340, f"Window: {row['window']}", 
            color='blue', rotation=90, ha='left', va='top', fontsize=12)

plt.savefig("ipi_plot.pdf")
print("Plot saved as 'ipi_plot.pdf")

## use trained data to test the hypothesis
### helper functions

In [ ]:
#################################################################
# 3. HELPER FUNCTIONS (NOW INCLUDES STOCHASTIC SAMPLING)
#################################################################

def sample_actions(logits_tuple):
    """
    Takes logits from the Actor and STOCHASTICALLY samples
    one action from each category.
    """
    move_logits, pull_logits, gaze_logits = logits_tuple
    
    # --- Move Action ---
    move_dist = Categorical(logits=move_logits)
    move_action = move_dist.sample() # Sampled action (e.g., tensor(2))
    
    # --- Pull Action ---
    pull_dist = Categorical(logits=pull_logits)
    pull_action = pull_dist.sample()
    
    # --- Gaze Action ---
    gaze_dist = Categorical(logits=gaze_logits)
    gaze_action = gaze_dist.sample()
    
    actions_dict = {
        'move': move_action.item(),
        'pull': pull_action.item(),
        'gaze': gaze_action.item()
    }
    
    # We don't need log_probs for analysis
    return actions_dict

### main testing functions

In [ ]:
def collect_data(num_steps_to_collect=10000):
    """
    Main function to load trained models and collect data.
    --- NOW USES STOCHASTIC SAMPLING ---
    --- *** FIXED: ADDED PULL MASKING *** ---
    """
    print(f"Loading models and setting up env for STOCHASTIC analysis...")
    
    # --- 1. Set up Environment ---
    env = CurriculumEnv(GRID_X, GRID_Y)
    env.set_training_phase("COOP_FULL", sync_window=20) # Set to final, 1s phase
    
    # --- 2. Load Trained Agents ---
    agent_A = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)
    agent_B = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)

    # --- *** MODIFIED: Load the NEW model files *** ---
    agent_A.load_state_dict(torch.load("agent_A_brain_new.pth"))
    agent_B.load_state_dict(torch.load("agent_B_brain_new.pth"))
    
    agent_A.eval()
    agent_B.eval()

    # --- 3. Initialize States ---
    state_A_dict, state_B_dict = env.reset()
    
    public_A = preprocess_state(state_A_dict, GRID_X, GRID_Y)
    public_B = preprocess_state(state_B_dict, GRID_X, GRID_Y)
    
    hidden_A = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)
    hidden_B = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)
    
    private_A = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
    private_B = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)

    # --- 4. Data Collection Loop ---
    analysis_data = [] 
    
    print(f"Running for {num_steps_to_collect} steps...")
    
    for step in range(num_steps_to_collect):
        if step % 1000 == 0:
            print(f"Step {step}/{num_steps_to_collect}...")
            
        with torch.no_grad():
            
            # -----------------------------------------------------------
            # --- *** NEW: CREATE ACTION MASKS (Same as train) *** ---
            # -----------------------------------------------------------
            at_lever_A = np.array_equal(state_A_dict['self_pos'], env.locations['lever_A'])
            at_lever_B = np.array_equal(state_B_dict['self_pos'], env.locations['lever_B'])

            can_pull_A = state_A_dict['can_pull']
            can_pull_B = state_B_dict['can_pull']

            if can_pull_A and at_lever_A:
                mask_A = torch.tensor([[0., 0.]], device=device).float()
            else:
                mask_A = torch.tensor([[0., -1e9]], device=device).float()
                
            if can_pull_B and at_lever_B:
                mask_B = torch.tensor([[0., 0.]], device=device).float()
            else:
                mask_B = torch.tensor([[0., -1e9]], device=device).float()
            # -----------------------------------------------------------
            # --- *** END NEW *** ---
            # -----------------------------------------------------------

            
            # --- Agent A runs (NOW PASSES MASK) ---
            (logits_A_tuple), value_A, next_hidden_A = agent_A(public_A, private_A, hidden_A, mask_A)
            actions_A_dict = sample_actions(logits_A_tuple)
            
            # --- Agent B runs (NOW PASSES MASK) ---
            (logits_B_tuple), value_B, next_hidden_B = agent_B(public_B, private_B, hidden_B, mask_B)
            actions_B_dict = sample_actions(logits_B_tuple) 

            # --- Environment steps ---
            next_state_A_dict, next_state_B_dict, reward_A, reward_B, done = \
                env.step(actions_A_dict, actions_B_dict)

            # --- Store Data (Removed 'is_drinking' as it's not in v16 state) ---
            analysis_data.append({
                'step': step,
                'A_pos_x': state_A_dict['self_pos'][0],
                'A_pos_y': state_A_dict['self_pos'][1],
                'A_hidden_state': hidden_A.detach().cpu().numpy().flatten(),
                'A_value': value_A.item(),
                'A_action_move': actions_A_dict['move'],
                'A_action_pull': actions_A_dict['pull'],
                'A_action_gaze': actions_A_dict['gaze'],
                'A_reward': reward_A,
                # 'A_is_drinking': state_A_dict['is_drinking'], # <-- Removed
                'B_pos_x': state_B_dict['self_pos'][0],
                'B_pos_y': state_B_dict['self_pos'][1],
                'B_hidden_state': hidden_B.detach().cpu().numpy().flatten(),
                'B_value': value_B.item(),
                'B_action_move': actions_B_dict['move'],
                'B_action_pull': actions_B_dict['pull'],
                'B_action_gaze': actions_B_dict['gaze'],
                'B_reward': reward_B,
                # 'B_is_drinking': state_B_dict['is_drinking'], # <-- Removed
                'lever_A_state': state_A_dict['self_lever_state'],
                'lever_B_state': state_B_dict['self_lever_state'],
                'port_A_state': state_A_dict['self_port_state'],
                'port_B_state': state_B_dict['self_port_state'],
                'success_sound': state_A_dict['success_sound']
            })

            # --- Prepare inputs for next step (Modified for new private input size) ---
            partner_pos_B_tensor = preprocess_partner_pos(env.agent_B_pos, GRID_X, GRID_Y)
            partner_pos_A_tensor = preprocess_partner_pos(env.agent_A_pos, GRID_X, GRID_Y)
            
            partner_A_can_pull = torch.tensor([[
                1.0 if next_state_A_dict['can_pull'] else 0.0
            ]], device=device).float()
            partner_B_can_pull = torch.tensor([[
                1.0 if next_state_B_dict['can_pull'] else 0.0
            ]], device=device).float()

            partner_A_time_norm = torch.tensor([[
                min(env.A_time_since_pull / env.current_sync_window, 1.0)
            ]], device=device).float()
            partner_B_time_norm = torch.tensor([[
                min(env.B_time_since_pull / env.current_sync_window, 1.0)
            ]], device=device).float()

            if actions_A_dict['gaze'] == 1:
                private_A = torch.cat([partner_pos_B_tensor, partner_B_can_pull, partner_B_time_norm], dim=-1)
            else:
                private_A = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
            
            if actions_B_dict['gaze'] == 1:
                private_B = torch.cat([partner_pos_A_tensor, partner_A_can_pull, partner_A_time_norm], dim=-1)
            else:
                private_B = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
            # --- End Gaze Tensor logic ---

            public_A = preprocess_state(next_state_A_dict, GRID_X, GRID_Y)
            public_B = preprocess_state(next_state_B_dict, GRID_X, GRID_Y)
            hidden_A = next_hidden_A
            hidden_B = next_hidden_B
            
            state_A_dict = next_state_A_dict
            state_B_dict = next_state_B_dict
            
    print("...Data collection complete.")
    return analysis_data


# --- To run the analysis ---\

if 1: # <-- Set to 1 to run analysis

    if __name__ == "__main__":

        # 1. Run the collection
        collected_data = collect_data(num_steps_to_collect=10000)

        # 2. Convert to Pandas DataFrame for analysis
        df = pd.DataFrame(collected_data)

        print(f"\nSuccessfully collected {len(df)} steps of data.")
        print("DataFrame head:")
        print(df.head())

        # 3. Save to a file (optional, but recommended)
        df.to_pickle("analysis_data_new.pkl")
        print("\nData saved to 'analysis_data_new.pkl'")

else:
    print('no testing')

In [ ]:
# Set the style for the plots
sns.set(style="darkgrid")
plt.rcParams['figure.figsize'] = (18, 7)

# --- 1. Load the Analysis Data ---
try:
    df = pd.read_pickle("analysis_data_new.pkl")
    print("Successfully loaded 'analysis_data_new.pkl'")
except FileNotFoundError:
    print("Error: 'analysis_data_new.pkl' not found.")
    print("Please run the 'collect_data()' function from the notebook to generate this file first.")
    # Create an empty DataFrame to prevent crashes, but plots will be empty
    df = pd.DataFrame() 

if not df.empty:

    # --- 2. Plot 1: Overall Success Rate (During Test Run) ---
    print("\n--- Generating Plot 1: Success Rate ---")
    
    # A "successful" pull is one that resulted in a large reward (> 5.0)
    # This indicates a coordinated pull.
    successful_A_pulls = df[(df['A_action_pull'] == 1) & (df['A_reward'] > 5.0)].shape[0]
    successful_B_pulls = df[(df['B_action_pull'] == 1) & (df['B_reward'] > 5.0)].shape[0]
    
    total_A_pulls = df[df['A_action_pull'] == 1].shape[0]
    total_B_pulls = df[df['B_action_pull'] == 1].shape[0]

    # Avoid division by zero if no pulls were attempted
    rate_A = (successful_A_pulls / total_A_pulls) if total_A_pulls > 0 else 0
    rate_B = (successful_B_pulls / total_B_pulls) if total_B_pulls > 0 else 0

    plt.figure()
    sns.barplot(
        x=['Agent A', 'Agent B'], 
        y=[rate_A, rate_B],
        palette=['#0072B2', '#D55E00'] # Blue, Orange
    )
    plt.title(f'Coordination Success Rate (Window=10, Steps={len(df)})', fontsize=16)
    plt.ylabel('Success Rate (Successful Pulls / Total Pulls)', fontsize=12)
    plt.ylim(0, 1.0)
    print(f"Agent A Success Rate: {rate_A:.2%}")
    print(f"Agent B Success Rate: {rate_B:.2%}")
    plt.savefig("analysis_success_rate.pdf")
    print("Saved 'analysis_success_rate.pdf'")
    plt.show()


    # --- 3. Plot 2: Internal Inter-Pull Interval (Within-Animal) ---
    print("\n--- Generating Plot 2: Internal IPI ---")
    
    # Find all attempted pulls for each agent
    df_A_pulls = df[df['A_action_pull'] == 1].copy()
    df_B_pulls = df[df['B_action_pull'] == 1].copy()

    # Calculate the time (steps) since its *own* last pull
    df_A_pulls['Internal_IPI'] = df_A_pulls['step'].diff()
    df_B_pulls['Internal_IPI'] = df_B_pulls['step'].diff()

    plt.figure()
    sns.histplot(df_A_pulls['Internal_IPI'].dropna(), bins=50, kde=True, color='#0072B2', label='Agent A: Internal IPI', stat='density')
    sns.histplot(df_B_pulls['Internal_IPI'].dropna(), bins=50, kde=True, color='#D55E00', label='Agent B: Internal IPI', stat='density')
    
    # Add a line for the "optimal" IPI (around 65 steps)
    plt.axvline(x=65, color='gray', linestyle=':', linewidth=3, label='Optimal IPI (approx. 65 steps)')
    
    plt.title('Internal Inter-Pull Interval (IPI) Distribution', fontsize=16)
    plt.xlabel('Steps Between Own Pulls', fontsize=12)
    plt.legend()
    plt.xlim(0, 200) # Zoom in on the main distribution
    plt.savefig("analysis_internal_ipi.pdf")
    print("Saved 'analysis_internal_ipi.pdf'")
    plt.show()


    # --- 4. Plot 3: Cross-Animal Inter-Pull Interval (Coordination) ---
    print("\n--- Generating Plot 3: Cross-Animal IPI (Corrected) ---")

    # We need the dataframes of pulls from Plot 2
    # If you don't have df_A_pulls and df_B_pulls, uncomment these lines:
    # df_A_pulls = df[df['A_action_pull'] == 1].copy()
    # df_B_pulls = df[df['B_action_pull'] == 1].copy()

    if 'df_A_pulls' in locals() and not df_A_pulls.empty:

        # Sort A's pulls by step and set as index
        df_A_pulls_sorted = df_A_pulls.sort_values('step').set_index('step')

        # Sort B's pulls but keep 'step' as a column
        df_B_pulls_with_step_col = df_B_pulls.sort_values('step').reset_index()

        # Find the nearest B pull for every A pull
        # This merges A (using its index) with B (using its 'step' column)
        df_merged = pd.merge_asof(
            df_A_pulls_sorted,
            df_B_pulls_with_step_col,
            left_index=True,
            right_on='step',  # Match A's index to B's 'step' column
            direction='nearest',
            suffixes=('_A', '_B')
        )

        # *** THIS IS THE CORRECTED LINE ***
        # The Cross-Animal IPI is the absolute difference between
        # Agent A's pull step (the index) and
        # Agent B's nearest pull step (the 'step' column)
        df_merged['Cross_IPI'] = (df_merged.index - df_merged['step']).abs()

        plt.figure()
        sns.histplot(df_merged['Cross_IPI'], bins=50, kde=False, color='#009E73') # Green

        # The environment window was 10
        plt.axvline(x=10, color='red', linestyle='--', linewidth=3, label='Success Window (10 steps)')

        plt.title('Cross-Animal Inter-Pull Interval (Coordination)', fontsize=16)
        plt.xlabel('Steps Between A-Pull and Nearest B-Pull', fontsize=12)
        plt.ylabel('Count of Coordinated Pulls', fontsize=12)
        plt.legend()
        plt.xlim(0, 100) # Zoom in
        plt.savefig("analysis_cross_animal_ipi.pdf")
        print("Saved 'analysis_cross_animal_ipi.pdf'")
        # plt.show() # Use plt.show() if in a non-notebook environment
        
        
        # --- 1. Plot 1: Successful vs. Failed Cross-Animal IPI ---
        print("\n--- Generating Plot 1: Successful vs. Failed Cross-Animal IPI ---")

        # Find all attempted pulls for each agent
        df_A_pulls = df[df['A_action_pull'] == 1].copy()
        df_B_pulls = df[df['B_action_pull'] == 1].copy()

        # Sort A's pulls by step and set as index
        df_A_pulls_sorted = df_A_pulls.sort_values('step').set_index('step')

        # Sort B's pulls but keep 'step' as a column
        df_B_pulls_with_step_col = df_B_pulls.sort_values('step').reset_index()

        # Find the nearest B pull for every A pull (same as corrected code)
        df_merged = pd.merge_asof(
            df_A_pulls_sorted,
            df_B_pulls_with_step_col,
            left_index=True,
            right_on='step',
            direction='nearest',
            suffixes=('_A', '_B')
        )

        # Calculate the Cross-IPI
        df_merged['Cross_IPI'] = (df_merged.index - df_merged['step']).abs()

        # *** NEW LOGIC: Define "Success" ***
        # Success is 1 if IPI is within the 10-step window, 0 otherwise
        df_merged['Success'] = (df_merged['Cross_IPI'] <= 10).astype(int)
        df_merged['Success_Label'] = df_merged['Success'].map({1: 'Successful Pulls', 0: 'Failed Pulls'})

        plt.figure(figsize=(18, 7))
        # Use 'hue' to split the data, 'stat' to show counts
        sns.histplot(
            data=df_merged,
            x='Cross_IPI',
            hue='Success_Label',
            multiple='stack', # Stack 'success' and 'fail' bars
            bins=50,
            palette={'Successful Pulls': '#009E73', 'Failed Pulls': '#D55E00'}
        )

        plt.axvline(x=10, color='red', linestyle='--', linewidth=3, label='Success Window (10 steps)')
        plt.title('Cross-Animal IPI (Split by Success)', fontsize=16)
        plt.xlabel('Steps Between A-Pull and Nearest B-Pull', fontsize=12)
        plt.legend()
        plt.xlim(0, 100) # Zoom in
        plt.savefig("analysis_cross_animal_ipi_split.pdf")
        print("Saved 'analysis_cross_animal_ipi_split.pdf'")
        # plt.show()


        # --- 2. Plot 2: Gaze Peri-Event Time Histogram (PETH) ---
        print("\n--- Generating Plot 2: Gaze Peri-Event Time Histogram ---")

        # Helper function to calculate the PETH
        def get_peth(event_series, action_series, window_size=4):
            """
            Calculates the average 'action' (e.g., gaze) around an 'event' (e.g., pull).
            """
            # Get the step index for all events (e.g., all pulls)
            event_steps = event_series[event_series == 1].index
            peth_data = []
            time_bins = list(range(-window_size, window_size + 1))

            for step in event_steps:
                for t_bin in time_bins:
                    target_step = step + t_bin
                    # Check if this step is in the dataframe's index
                    if target_step in action_series.index:
                        # Append the time bin and the action (gaze) that happened
                        peth_data.append({
                            'time_bin': t_bin,
                            'gaze_action': action_series.get(target_step, 0) # Default to 0 if out of bounds
                        })

            if not peth_data:
                return pd.DataFrame(columns=['time_bin', 'gaze_probability'])

            peth_df = pd.DataFrame(peth_data)

            # Average the gaze action for each time bin
            peth_summary = peth_df.groupby('time_bin')['gaze_action'].mean().reset_index()
            peth_summary.rename(columns={'gaze_action': 'gaze_probability'}, inplace=True)
            return peth_summary

        # Set the time window
        time_window = 40

        # --- Calculate the four PETHs ---
        # 1. Agent A's gaze around Agent A's OWN pull
        peth_A_self = get_peth(df['A_action_pull'], df['A_action_gaze'], time_window)
        # 2. Agent B's gaze around Agent B's OWN pull
        peth_B_self = get_peth(df['B_action_pull'], df['B_action_gaze'], time_window)

        # 3. Agent A's gaze around Agent B's (PARTNER) pull
        peth_A_partner = get_peth(df['B_action_pull'], df['A_action_gaze'], time_window)
        # 4. Agent B's gaze around Agent A's (PARTNER) pull
        peth_B_partner = get_peth(df['A_action_pull'], df['B_action_gaze'], time_window)

        # --- Create the plots ---
        fig, axes = plt.subplots(1, 2, figsize=(20, 7), sharey=True)

        # Plot 2a: Gaze around SELF Pull
        sns.lineplot(data=peth_A_self, x='time_bin', y='gaze_probability', ax=axes[0], label='Agent A', color='#0072B2', linewidth=2.5)
        sns.lineplot(data=peth_B_self, x='time_bin', y='gaze_probability', ax=axes[0], label='Agent B', color='#D55E00', linewidth=2.5)
        axes[0].axvline(x=0, color='red', linestyle='--', label='Self Pull Event')
        axes[0].set_title('Gaze Probability Around OWN Pull', fontsize=16)
        axes[0].set_xlabel('Time (steps) Relative to Own Pull', fontsize=12)
        axes[0].set_ylabel('Gaze Probability', fontsize=12)
        axes[0].legend()

        # Plot 2b: Gaze around PARTNER Pull
        sns.lineplot(data=peth_A_partner, x='time_bin', y='gaze_probability', ax=axes[1], label='Agent A (gazing)', color='#0072B2', linewidth=2.5)
        sns.lineplot(data=peth_B_partner, x='time_bin', y='gaze_probability', ax=axes[1], label='Agent B (gazing)', color='#D55E00', linewidth=2.5)
        axes[1].axvline(x=0, color='red', linestyle='--', label='Partner Pull Event')
        axes[1].set_title('Gaze Probability Around PARTNER\'s Pull', fontsize=16)
        axes[1].set_xlabel('Time (steps) Relative to Partner\'s Pull', fontsize=12)
        axes[1].set_ylabel('Gaze Probability', fontsize=12)
        axes[1].legend()

        plt.suptitle('Peri-Event Time Histogram (PETH) for Gaze Actions', fontsize=20)
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.savefig("analysis_gaze_peth.pdf")
        print("Saved 'analysis_gaze_peth.pdf'")
        # plt.show()

In [ ]:
np.unique(df['A_reward'])

In [ ]:
np.unique(df['B_reward'])